# Practical 10 – Association Mining

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


## Load the transaction dataset

The source file has no header row, so it is read with `header=None`.

Each row represents one transaction and each non-empty cell represents an item in that transaction.

The local/GitHub fallback keeps the notebook portable between Anaconda/Jupyter and Colab.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = "Dataset.csv"
LOCAL_CANDIDATES = [
    Path("data") / DATA_FILE,
    Path("Lecture10_Association_Mining") / "data" / DATA_FILE
]
GITHUB_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture10_Association_Mining/data/Dataset.csv"

local_path = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
data_source = local_path if local_path else GITHUB_URL

# The source chapter treats every row as a transaction, so there is no header row.
transactions_raw = pd.read_csv(data_source, header=None)

print(f"Loaded from: {data_source}")
transactions_raw.head()

In [ ]:
# If mlxtend is missing in Colab, run: !pip install mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

transactions=[]
for _,row in transactions_raw.iterrows():
    items=[str(x).strip() for x in row.dropna().tolist() if str(x).strip()]
    transactions.append(items)

te=TransactionEncoder()
encoded=te.fit(transactions).transform(transactions)
basket=pd.DataFrame(encoded,columns=te.columns_)
basket.head()

In [ ]:
# Source-chapter thresholds:
# minimum support = 0.249
# minimum confidence = 0.699

frequent = apriori(
    basket,
    min_support=0.249,
    use_colnames=True
)

rules = association_rules(
    frequent,
    metric="confidence",
    min_threshold=0.699
)

display(
    frequent.sort_values("support", ascending=False)
)

display(
    rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].sort_values("confidence", ascending=False)
)

## Exercise
Change minimum support and confidence and explain how the number of rules changes.

## Source-chapter alignment

The original author code uses the `apriori_python` package with:

- `minSup = 0.249`
- `minConf = 0.699`

This practical uses `mlxtend` instead, but now applies the same support and confidence thresholds and the same transaction dataset. The implementation library differs; the worked association-mining example is equivalent.